# 03 · Treino com MLflow

*Popularidade × filtragem colaborativa × fatoração de matriz*

> **FIAP Pós-Tech · Fase 5 (MLOps)** — Recomendação de notícias com MLOps no Databricks.
> Matérias **reais** (sitemap do O Globo) · usuários e cliques **sintéticos**.

## Três jeitos de recomendar

| Modelo | Ideia | Pergunta que responde |
|---|---|---|
| **Popularidade** (baseline) | o que todo mundo está lendo agora | "o que está bombando?" |
| **Item-item CF** | quem leu X também leu Y (cosseno entre colunas da matriz usuário×matéria) | "parecido com o que você leu" |
| **Fatoração de matriz (SVD)** | usuários e matérias num espaço de *gostos latentes* | "gente com seu gosto leu..." |

O baseline é obrigatório: **se o modelo sofisticado não ganha da popularidade, ele não
se paga.** Todos treinam no mesmo split e são medidos com as mesmas métricas.

## Como avaliar um recomendador (offline)

- **Split temporal**: para cada usuário, os 20% cliques mais recentes viram teste.
  Nunca aleatório — em produção o modelo sempre prevê o *futuro*.
- **Precision@10**: dos 10 recomendados, quantos ele leu?
- **Recall@10**: do que ele leu, quanto estava nos 10?
- **NDCG@10**: como o Recall, mas acertar no **topo** da lista vale mais. ← *métrica de promoção*
- **HitRate@10**: acertou pelo menos um?
- **Cobertura**: que fração do catálogo aparece em alguma recomendação (diversidade).

In [ ]:
# Célula 1 — dependências (sempre a primeira: o restartPython apaga as variáveis)
%pip install -q -r ../requirements-databricks.txt
dbutils.library.restartPython()

In [ ]:
# Célula 2 — src/ no sys.path + widgets (= parâmetros do Job) -> settings
import sys
from pathlib import Path


def _find_src() -> Path:
    candidates = [Path.cwd().parent / "src"]
    try:  # fallback: caminho do notebook no workspace
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        candidates.append(Path("/Workspace" + ctx.notebookPath().get()).parent.parent / "src")
    except Exception:  # noqa: BLE001
        pass
    for c in candidates:
        if (c / "news_recsys").is_dir():
            return c.resolve()
    raise FileNotFoundError(f"src/ não encontrado em {candidates}")


SRC = _find_src()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
ROOT = SRC.parent

from news_recsys.config import Settings

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "news_recsys_dev")
dbutils.widgets.text("n_users", "300")
dbutils.widgets.text("seed", "42")

settings = Settings(_env_file=None).with_overrides(
    catalog=dbutils.widgets.get("catalog"),
    schema_name=dbutils.widgets.get("schema"),
    n_users=dbutils.widgets.get("n_users"),
    seed=dbutils.widgets.get("seed"),
)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {settings.catalog}.{settings.schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {settings.catalog}.{settings.schema_name}.{settings.volume}")
print(f"schema ..... {settings.catalog}.{settings.schema_name}")
print(f"modelo ..... {settings.model_fqn}")
print(f"usuários ... {settings.n_users} (seed={settings.seed})")

In [ ]:
# MLflow: experimento por usuário + registro de modelos no Unity Catalog
import mlflow
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
ME = w.current_user.me().user_name
EXPERIMENT = f"/Users/{ME}/news_recsys_{settings.schema_name}"
mlflow.set_experiment(EXPERIMENT)
mlflow.set_registry_uri("databricks-uc")
print("experimento:", EXPERIMENT)

In [ ]:
# Dados de entrada: catálogo (bronze) + usuários e cliques (silver)
import pandas as pd

articles = spark.table(settings.table("bronze_articles")).toPandas()
window = spark.table(settings.table("silver_catalog_window")).toPandas()["article_id"]
articles = articles[articles["article_id"].isin(window)].reset_index(drop=True)
users = spark.table(settings.table("silver_users")).toPandas()
clicks = spark.table(settings.table("silver_interactions")).toPandas()
clicks["event_ts"] = pd.to_datetime(clicks["event_ts"], utc=True)
print(f"{len(articles)} matérias · {len(users)} usuários · {len(clicks)} cliques")

### `models/base.py` — o contrato comum

In [ ]:
# src/news_recsys/models/base.py  (somente leitura — o código de verdade está em src/)
"""Base comum dos recomendadores: índices, matriz usuário×matéria e o ``recommend`` top-k."""

from __future__ import annotations

from abc import ABC, abstractmethod
from collections.abc import Sequence
from typing import Any

import numpy as np
import pandas as pd
from scipy import sparse

RECS_COLUMNS = ["user_id", "rank", "article_id", "score"]


class NotFittedError(RuntimeError):
    """``recommend`` chamado antes de ``fit``."""


class Recommender(ABC):
    """Contrato: ``fit(cliques, catálogo)`` → ``recommend(usuários, k)``.

    O catálogo inteiro é candidato (inclusive matérias novas, sem cliques). Matérias já lidas
    pelo usuário não são recomendadas de novo. Usuários desconhecidos recebem o *fallback* de
    popularidade — o "cold start" de usuário.
    """

    name: str = "base"

    def __init__(self) -> None:
        self.item_ids: np.ndarray = np.array([], dtype=object)
        self.user_ids: np.ndarray = np.array([], dtype=object)
        self._item_index: dict[str, int] = {}
        self._user_index: dict[str, int] = {}
        self.X: sparse.csr_matrix | None = None
        self.popularity_: np.ndarray = np.array([])
        self.fitted_ = False

    # --- API pública ------------------------------------------------------------------
    def fit(self, interactions: pd.DataFrame, articles: pd.DataFrame) -> Recommender:
        self.item_ids = articles["article_id"].astype(str).to_numpy()
        self._item_index = {a: i for i, a in enumerate(self.item_ids)}
        known = interactions[interactions["article_id"].isin(self._item_index)]
        self.user_ids = np.array(sorted(known["user_id"].astype(str).unique()), dtype=object)
        self._user_index = {u: i for i, u in enumerate(self.user_ids)}

        rows = known["user_id"].map(self._user_index).to_numpy()
        cols = known["article_id"].map(self._item_index).to_numpy()
        data = np.ones(len(known), dtype=np.float32)
        self.X = sparse.csr_matrix(
            (data, (rows, cols)), shape=(len(self.user_ids), len(self.item_ids))
        )
        self.X.sum_duplicates()
        counts = np.asarray(self.X.sum(axis=0)).ravel()
        self.popularity_ = counts / counts.max() if counts.max() > 0 else counts
        self._fit(known, articles)
        self.fitted_ = True
        return self

    def recommend(
        self, user_ids: Sequence[str], k: int = 10, exclude_seen: bool = True
    ) -> pd.DataFrame:
        if not self.fitted_ or self.X is None:
            raise NotFittedError(f"{self.name}: chame fit() antes de recommend()")
        users = [str(u) for u in user_ids]
        known = [u for u in users if u in self._user_index]
        rows: list[dict[str, Any]] = []

        if known:
            idx = np.array([self._user_index[u] for u in known])
            scores = np.asarray(self._score(idx), dtype=float)
            scores = scores + 1e-6 * self.popularity_  # desempate (e fallback) por popularidade
            if exclude_seen:
                seen = self.X[idx].toarray() > 0
                scores[seen] = -np.inf
            for u, row in zip(known, scores, strict=True):
                rows.extend(self._top_k(u, row, k))

        cold_scores = self._cold_start_scores()
        for u in users:
            if u not in self._user_index:
                rows.extend(self._top_k(u, cold_scores, k))

        out = pd.DataFrame(rows, columns=RECS_COLUMNS)
        order = {u: i for i, u in enumerate(users)}
        out["_o"] = out["user_id"].map(order)
        return out.sort_values(["_o", "rank"]).drop(columns="_o").reset_index(drop=True)

    def params(self) -> dict[str, Any]:
        """Hiperparâmetros (logados no MLflow)."""
        return {}

    # --- Para as subclasses ------------------------------------------------------------
    def _fit(self, interactions: pd.DataFrame, articles: pd.DataFrame) -> None:  # noqa: B027
        """Treino específico do modelo (opcional)."""

    def _cold_start_scores(self) -> np.ndarray:
        """Scores para usuários sem histórico (cold start): popularidade."""
        return self.popularity_.astype(float)

    @abstractmethod
    def _score(self, user_idx: np.ndarray) -> np.ndarray:
        """Matriz (n_usuários × n_matérias) de scores."""

    # --- Internos -------------------------------------------------------------------------
    def _top_k(self, user_id: str, scores: np.ndarray, k: int) -> list[dict[str, Any]]:
        finite = np.isfinite(scores)
        k_eff = min(k, int(finite.sum()))
        if k_eff == 0:
            return []
        candidates = np.where(finite)[0]
        part = candidates[np.argpartition(-scores[candidates], k_eff - 1)[:k_eff]]
        best = part[np.argsort(-scores[part], kind="stable")]
        return [
            {
                "user_id": user_id,
                "rank": r,
                "article_id": str(self.item_ids[i]),
                "score": float(scores[i]),
            }
            for r, i in enumerate(best, start=1)
        ]

### `models/popularity.py`, `models/item_cf.py`, `models/svd.py`

In [ ]:
# src/news_recsys/models/popularity.py  (somente leitura — o código de verdade está em src/)
"""Baseline: as matérias mais clicadas (com bônus de recência). Todo recsys começa aqui."""

from __future__ import annotations

from typing import Any

import numpy as np
import pandas as pd

from news_recsys.models.base import Recommender


class PopularityRecommender(Recommender):
    name = "popularidade"

    def __init__(self, recency_half_life_h: float | None = 24.0) -> None:
        super().__init__()
        self.recency_half_life_h = recency_half_life_h
        self._scores = np.array([])

    def _fit(self, interactions: pd.DataFrame, articles: pd.DataFrame) -> None:
        scores = self.popularity_.astype(float)
        if self.recency_half_life_h:
            published = pd.to_datetime(
                articles["published_at"], utc=True, format="mixed", errors="coerce"
            )
            ref = published.max()
            age_h = ((ref - published).dt.total_seconds() / 3600).fillna(1e6).to_numpy()
            scores = scores * 0.5 ** (np.clip(age_h, 0, None) / self.recency_half_life_h)
        self._scores = scores

    def _cold_start_scores(self) -> np.ndarray:
        return self._scores.astype(float)

    def _score(self, user_idx: np.ndarray) -> np.ndarray:
        return np.tile(self._scores, (len(user_idx), 1))

    def params(self) -> dict[str, Any]:
        return {"recency_half_life_h": self.recency_half_life_h}

In [ ]:
# src/news_recsys/models/item_cf.py  (somente leitura — o código de verdade está em src/)
"""Filtragem colaborativa item-item: "quem leu X também leu Y" (similaridade de cosseno)."""

from __future__ import annotations

from typing import Any

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.preprocessing import normalize

from news_recsys.models.base import Recommender


class ItemCFRecommender(Recommender):
    name = "item_cf"

    def __init__(self, n_neighbors: int = 50) -> None:
        super().__init__()
        self.n_neighbors = n_neighbors
        self.S: sparse.csr_matrix | None = None

    def _fit(self, interactions: pd.DataFrame, articles: pd.DataFrame) -> None:
        assert self.X is not None
        Xn = normalize(self.X.tocsc(), norm="l2", axis=0)  # cada matéria vira um vetor unitário
        S = (Xn.T @ Xn).tocsr()
        S.setdiag(0)
        S.eliminate_zeros()
        self.S = self._prune(S, self.n_neighbors)

    @staticmethod
    def _prune(S: sparse.csr_matrix, n: int) -> sparse.csr_matrix:
        """Mantém só os ``n`` vizinhos mais parecidos de cada matéria."""
        S = S.tolil()
        for i, (cols, vals) in enumerate(zip(S.rows, S.data, strict=True)):
            if len(vals) > n:
                keep = np.argsort(vals)[::-1][:n]
                S.rows[i] = [cols[j] for j in sorted(keep)]
                S.data[i] = [vals[j] for j in sorted(keep)]
        return S.tocsr()

    def _score(self, user_idx: np.ndarray) -> np.ndarray:
        assert self.X is not None and self.S is not None
        return (self.X[user_idx] @ self.S).toarray()

    def params(self) -> dict[str, Any]:
        return {"n_neighbors": self.n_neighbors}

In [ ]:
# src/news_recsys/models/svd.py  (somente leitura — o código de verdade está em src/)
"""Fatoração de matriz (TruncatedSVD): usuários e matérias num espaço de "gostos" latentes."""

from __future__ import annotations

from typing import Any

import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD

from news_recsys.models.base import Recommender


class SVDRecommender(Recommender):
    name = "svd"

    def __init__(self, n_components: int = 24, random_state: int = 42) -> None:
        super().__init__()
        self.n_components = n_components
        self.random_state = random_state
        self.user_factors_: np.ndarray = np.array([])
        self.item_factors_: np.ndarray = np.array([])

    def _fit(self, interactions: pd.DataFrame, articles: pd.DataFrame) -> None:
        assert self.X is not None
        n_comp = max(1, min(self.n_components, min(self.X.shape) - 1))
        svd = TruncatedSVD(n_components=n_comp, random_state=self.random_state)
        self.user_factors_ = svd.fit_transform(self.X)  # U·Σ
        self.item_factors_ = svd.components_  # Vᵀ
        self.explained_variance_ = float(svd.explained_variance_ratio_.sum())

    def _score(self, user_idx: np.ndarray) -> np.ndarray:
        return self.user_factors_[user_idx] @ self.item_factors_

    def params(self) -> dict[str, Any]:
        return {"n_components": self.n_components, "random_state": self.random_state}

### `evaluation/split.py`, `evaluation/metrics.py` e `training.py`

In [ ]:
# src/news_recsys/evaluation/metrics.py  (somente leitura — o código de verdade está em src/)
"""Métricas de ranking (Precision/Recall/NDCG/HitRate @K) e cobertura de catálogo."""

from __future__ import annotations

from collections.abc import Sequence
from typing import Any

import numpy as np
import pandas as pd


def precision_at_k(recommended: Sequence[str], relevant: set[str], k: int) -> float:
    if k <= 0:
        return 0.0
    return sum(1 for a in list(recommended)[:k] if a in relevant) / k


def recall_at_k(recommended: Sequence[str], relevant: set[str], k: int) -> float:
    if not relevant or k <= 0:
        return 0.0
    return len(set(list(recommended)[:k]) & relevant) / len(relevant)


def hit_rate_at_k(recommended: Sequence[str], relevant: set[str], k: int) -> float:
    return 1.0 if set(list(recommended)[:k]) & relevant else 0.0


def ndcg_at_k(recommended: Sequence[str], relevant: set[str], k: int) -> float:
    """Ganho cumulativo descontado: acertar no topo vale mais que acertar no fim da lista."""
    if not relevant or k <= 0:
        return 0.0
    dcg = sum(1.0 / np.log2(i + 2) for i, a in enumerate(list(recommended)[:k]) if a in relevant)
    ideal = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return float(dcg / ideal) if ideal else 0.0


def evaluate_recommendations(
    recs: pd.DataFrame, test: pd.DataFrame, k: int, catalog_size: int
) -> dict[str, float]:
    """Média das métricas por usuário do teste + cobertura de catálogo."""
    relevant = test.groupby("user_id")["article_id"].apply(set).to_dict()
    by_user = (
        recs.sort_values(["user_id", "rank"]).groupby("user_id")["article_id"].apply(list).to_dict()
    )
    rows = []
    for user, rel in relevant.items():
        rec = by_user.get(user, [])
        rows.append(
            (
                precision_at_k(rec, rel, k),
                recall_at_k(rec, rel, k),
                ndcg_at_k(rec, rel, k),
                hit_rate_at_k(rec, rel, k),
            )
        )
    arr = np.array(rows) if rows else np.zeros((1, 4))
    n_distinct = recs["article_id"].nunique() if len(recs) else 0
    return {
        "precision_at_k": float(arr[:, 0].mean()),
        "recall_at_k": float(arr[:, 1].mean()),
        "ndcg_at_k": float(arr[:, 2].mean()),
        "hit_rate_at_k": float(arr[:, 3].mean()),
        "catalog_coverage": n_distinct / catalog_size if catalog_size else 0.0,
        "n_users_eval": float(len(rows)),
    }


def evaluate_model(
    model: Any, test: pd.DataFrame, k: int, catalog_size: int
) -> tuple[dict[str, float], pd.DataFrame]:
    """Recomenda para os usuários do teste e mede. Retorna (métricas, recomendações)."""
    users = sorted(test["user_id"].unique())
    recs = model.recommend(users, k=k)
    return evaluate_recommendations(recs, test, k, catalog_size), recs

In [ ]:
# src/news_recsys/training.py  (somente leitura — o código de verdade está em src/)
"""Pipeline de treino puro (sem MLflow): split → fit → avaliação de cada candidato.

Os notebooks embrulham isto com MLflow; os testes e o ``make train-local`` usam direto.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any

import pandas as pd

from news_recsys.evaluation.metrics import evaluate_model
from news_recsys.evaluation.split import temporal_split
from news_recsys.models import MODEL_REGISTRY
from news_recsys.models.base import Recommender

DEFAULT_CANDIDATES: list[tuple[str, dict[str, Any]]] = [
    ("popularidade", {"recency_half_life_h": 24.0}),
    ("item_cf", {"n_neighbors": 50}),
    ("svd", {"n_components": 16}),
    ("svd", {"n_components": 32}),
]


@dataclass
class CandidateResult:
    name: str
    params: dict[str, Any]
    metrics: dict[str, float]
    model: Recommender = field(repr=False)

    @property
    def label(self) -> str:
        suffix = ",".join(f"{k}={v}" for k, v in self.params.items())
        return f"{self.name}({suffix})"


def build_model(name: str, params: dict[str, Any]) -> Recommender:
    if name not in MODEL_REGISTRY:
        raise KeyError(f"modelo desconhecido: {name} (opções: {sorted(MODEL_REGISTRY)})")
    return MODEL_REGISTRY[name](**params)


def train_candidates(
    interactions: pd.DataFrame,
    articles: pd.DataFrame,
    candidates: list[tuple[str, dict[str, Any]]] | None = None,
    k: int = 10,
    test_fraction: float = 0.2,
) -> list[CandidateResult]:
    """Treina e avalia cada candidato no MESMO split (comparação justa)."""
    train, test = temporal_split(interactions, test_fraction=test_fraction)
    results = []
    for name, params in candidates or DEFAULT_CANDIDATES:
        model = build_model(name, params).fit(train, articles)
        metrics, _ = evaluate_model(model, test, k=k, catalog_size=len(articles))
        results.append(CandidateResult(name, params, metrics, model))
    return results


def best_candidate(results: list[CandidateResult], metric: str = "ndcg_at_k") -> CandidateResult:
    return max(results, key=lambda r: r.metrics[metric])


def refit_full(
    result: CandidateResult, interactions: pd.DataFrame, articles: pd.DataFrame
) -> Recommender:
    """Depois de escolher, re-treina com TODOS os cliques (o modelo de produção)."""
    return build_model(result.name, result.params).fit(interactions, articles)

### Execução: treinando os candidatos — uma *run* do MLflow para cada

Uma run "pai" agrupa o treino do dia; cada candidato vira uma run "filha" com seus
parâmetros e métricas. Depois abra **Experiments** no menu e compare lado a lado.

In [ ]:
from datetime import UTC, datetime

from news_recsys.training import DEFAULT_CANDIDATES, best_candidate, refit_full, train_candidates

run_name = f"treino-{datetime.now(UTC):%Y%m%d-%H%M}"
parent = mlflow.start_run(run_name=run_name)
mlflow.log_params({"n_users": settings.n_users, "seed": settings.seed, "n_articles": len(articles), "n_clicks": len(clicks), "k": settings.k})

results = train_candidates(clicks, articles, DEFAULT_CANDIDATES, k=settings.k, test_fraction=settings.test_fraction)
for r in results:
    with mlflow.start_run(run_name=r.label, nested=True):
        mlflow.log_param("model", r.name)
        mlflow.log_params(r.params)
        mlflow.log_metrics(r.metrics)

comparison = pd.DataFrame([{"candidato": r.label, **r.metrics} for r in results]).set_index("candidato")
comparison.drop(columns="n_users_eval").sort_values("ndcg_at_k", ascending=False).round(4)

### Visualizando a comparação

In [ ]:
import matplotlib.pyplot as plt

ax = comparison[["ndcg_at_k", "recall_at_k", "hit_rate_at_k"]].sort_values("ndcg_at_k").plot.barh(figsize=(8, 3.8))
ax.set_title("Candidatos — avaliação offline (@10)")
ax.set_xlabel("valor")
plt.tight_layout()
display(ax.figure)

### Execução: o vencedor é re-treinado com TODOS os dados e registrado

Escolhemos pelo NDCG@10, re-treinamos com todos os cliques (o teste era só para medir)
e logamos como **modelo MLflow (pyfunc)** com assinatura e exemplo de entrada.
`registered_model_name` cria uma **nova versão** no Unity Catalog — que ainda **não**
é a campeã: quem decide é o notebook 04.

In [ ]:
# src/news_recsys/pyfunc_model.py  (somente leitura — o código de verdade está em src/)
"""Wrapper MLflow (pyfunc) do recomendador — o artefato registrado no Unity Catalog.

Único módulo que importa MLflow no topo: ele É o modelo. Os testes não o importam.
Entrada: DataFrame com ``user_id`` (e opcionalmente ``k``). Saída: user_id, rank, article_id, score.
"""

from __future__ import annotations

from typing import Any

import mlflow
import pandas as pd

from news_recsys.models.base import RECS_COLUMNS, Recommender

DEFAULT_K = 10


class NewsRecommenderModel(mlflow.pyfunc.PythonModel):
    def __init__(self, recommender: Recommender, k: int = DEFAULT_K) -> None:
        self.recommender = recommender
        self.k = k

    def predict(
        self, context: Any, model_input: pd.DataFrame, params: dict[str, Any] | None = None
    ) -> pd.DataFrame:
        k = int((params or {}).get("k", self.k))
        users = model_input["user_id"].astype(str).tolist()
        recs = self.recommender.recommend(users, k=k)
        return recs[RECS_COLUMNS]


def log_recommender(
    recommender: Recommender,
    *,
    code_path: str,
    sample_users: list[str],
    registered_model_name: str | None = None,
    k: int = DEFAULT_K,
) -> Any:
    """Loga o modelo com assinatura e exemplo; registra no UC se ``registered_model_name``."""
    from mlflow.models import infer_signature

    wrapper = NewsRecommenderModel(recommender, k=k)
    example = pd.DataFrame({"user_id": sample_users[:3]})
    output = wrapper.predict(None, example)
    signature = infer_signature(example, output, params={"k": k})
    return mlflow.pyfunc.log_model(
        name="news_recommender",
        python_model=wrapper,
        code_paths=[code_path],
        signature=signature,
        input_example=example,
        registered_model_name=registered_model_name,
        pip_requirements=["pandas", "numpy", "scipy", "scikit-learn", "pydantic-settings"],
    )

In [ ]:
from news_recsys.pyfunc_model import log_recommender

best = best_candidate(results, settings.promotion_metric)
production_model = refit_full(best, clicks, articles)

mlflow.log_param("winner", best.label)
mlflow.log_metrics(best.metrics)  # métricas do vencedor na run pai (usadas pelo quality gate)
model_info = log_recommender(
    production_model,
    code_path=str(SRC / "news_recsys"),
    sample_users=users["user_id"].tolist(),
    registered_model_name=settings.model_fqn,
    k=settings.k,
)
run_id = parent.info.run_id
mlflow.end_run()

version = str(model_info.registered_model_version)
print(f"vencedor: {best.label}  |  NDCG@10 = {best.metrics['ndcg_at_k']:.4f}")
print(f"registrado: {settings.model_fqn} versão {version}  (run {run_id})")

# Passa a versão para a próxima task do Job (04_avaliacao_registro)
dbutils.jobs.taskValues.set(key="model_version", value=version)
dbutils.jobs.taskValues.set(key="run_id", value=run_id)

### Execução: o modelo registrado funciona fora deste notebook?

In [ ]:
loaded = mlflow.pyfunc.load_model(f"models:/{settings.model_fqn}/{version}")
loaded.predict(pd.DataFrame({"user_id": users["user_id"].head(2)})).merge(
    articles[["article_id", "section", "title"]], on="article_id"
)

### Dúvida comum: e a matéria que saiu há 5 minutos? (*cold start*)

A filtragem colaborativa só recomenda o que **já foi clicado** — uma matéria nova não
tem cliques, então não aparece. Em notícias isso é crítico. Soluções de mercado:
misturar com popularidade recente (nosso desempate), modelos de **conteúdo** (texto,
editoria) ou híbridos. Usuário novo, sem histórico, recebe a lista de popularidade.

### O que percebemos?

- O baseline perde feio — personalização **vale a pena** (com estes dados).
- O MLflow guardou tudo: parâmetros, métricas, o artefato e de onde ele veio (*lineage*).
- Registrar ≠ publicar: a versão nova espera o *quality gate*.

**Próximo:** `04_avaliacao_registro`.